# Trabajo Práctico Final: Simulación de Eventos Discretos
## Estudio de la eficiencia técnica, operativa y financiera en la infraestructura de una estación de carga de vehículos eléctricos a través de la simulación de eventos discretos en CABA

**Equipo:** Carlana Rivero, Loglen, Millán, Ojeda Cabrera.

**Objetivo:** Simular, mediante la metodología evento a evento, el funcionamiento de una red de estaciones de carga de vehículos eléctricos en CABA con expansión dinámica de la infraestructura, demanda variable por franja horaria y a largo plazo, fallas y mantenimiento de cargadores, y estructura de costos dinámica. Se busca determinar los valores de las variables de control (`RC`, `TMP`, `PDCE`) que equilibren congestión, arrepentimiento, capacidad ociosa y beneficio económico.

> Las celdas marcadas **[DESACTUALIZADO — TP 4]** son copia del TP 4 y deben actualizarse al modelo del TP Final. Las marcadas **[PENDIENTE]** son esqueleto sin implementar.

In [ ]:
# [DESACTUALIZADO — TP 4] verificar la ruta de Drive del TP Final al cargar los datos.
from google.colab import drive
# Montaje de Google Drive para el acceso a los datasets históricos
drive.mount('/content/drive')

## 1. Configuración del Entorno e Importación de Librerías

Se procede con la preparación del entorno de ejecución, incluyendo la instalación de la librería `fitter` para el modelado estadístico y la importación de módulos para el procesamiento de datos, visualización y simulación estocástica.

> **[DESACTUALIZADO — TP 4]** Falta el import de `dataclasses` (estado del motor). Verificar si la versión instalada de `fitter` expone `n_jobs` en `fit()`.

In [ ]:
pip install fitter

In [ ]:
# [DESACTUALIZADO — TP 4] agregar los imports del TP Final (ver nota de arriba).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import curve_fit, OptimizeWarning
from fitter import Fitter
import math
import random
import warnings
from typing import List

## 2. Ajuste de la Curva de Crecimiento del Parque Automotor Eléctrico

La demanda de carga no es estacionaria en el largo plazo: el parque de vehículos eléctricos (BEV) e híbridos enchufables (PHEV) crece de forma acelerada hasta un punto de inflexión, a partir del cual se desacelera hasta saturar. En esta sección se reconstruye la evolución de ese parque en CABA, se elige la curva que proyecta su crecimiento de forma más realista, se estima su techo a partir de los países que ya atravesaron la etapa actual de CABA y se deriva de ella el factor variable en el tiempo que multiplica los intervalos entre arribos generados por el motor.

**Fuentes.** Los archivos están en la carpeta `Crecimiento del Parque/` del repositorio, que se copia a `Datos/Crecimiento del Parque/` en el Drive del TP Final:
- `Total Patentamientos CABA.xlsx`: patentamientos trimestrales de automotores por tipo y segmento en la Ciudad de Buenos Aires (Instituto de Estadística y Censos de la Ciudad, IDECBA, cuadro AC_AU_AX03, sobre datos de ACARA).
- `Distribución Patentamientos CABA.xlsx`: distribución porcentual trimestral de esos patentamientos por tipo de motor y combustible (IDECBA, cuadro AC_AU_AX08, sobre datos de ACARA).
- `EV data by country 2026.xlsx`: stock y ventas de autos eléctricos por país (IEA, *Global EV Outlook 2026*).
- `bachelor_Pham_Minh_Khoa_2023.pdf`: Pham (2023), referencia metodológica para el ajuste de curvas de crecimiento al parque eléctrico; no se lee desde el código.

Dos datos entran como constantes, porque provienen de informes y no de series: la proporción anual de enchufables entre los electrificados patentados en Argentina (informes de electromovilidad del SIOMAA) y el porcentaje de usuarios que carga en estaciones públicas ([IEA, *Global EV Outlook 2024*](https://iea.blob.core.windows.net/assets/a9e3544b-0b12-4e15-b407-65f5c8ce1b5f/GlobalEVOutlook2024.pdf), valor de México).

### 2.1. Serie histórica de CABA

Stock trimestral de autos enchufables que cargan en estaciones públicas, reconstruido a partir de los patentamientos del primer trimestre de 2020 al segundo de 2026.

In [ ]:
# Fuentes: carpeta "Crecimiento del Parque" del repositorio, copiada a la carpeta de datos del TP Final en Drive.
PATH_CRECIMIENTO = "/content/drive/MyDrive/Colab Notebooks/TP Final Simu/Datos/Crecimiento del Parque/"
# openpyxl avisa que descarta extensiones de formato de Excel que no soporta; no afecta los valores leídos.
warnings.filterwarnings("ignore", message="Unknown extension is not supported")

# Cuadros del IDECBA: tres filas de título y encabezado; cada año es una fila de cabecera seguida de sus trimestres.
patentamientos = pd.read_excel(PATH_CRECIMIENTO + "Total Patentamientos CABA.xlsx", sheet_name="AC_AU_AX03",
                               header=None, skiprows=3, usecols="A:C", names=["periodo", "total", "autos"])
distribucion = pd.read_excel(PATH_CRECIMIENTO + "Distribución Patentamientos CABA.xlsx", sheet_name="AC_AU_AX08",
                             header=None, skiprows=3, usecols="A,G", names=["periodo", "pct_electrificados"])
iea = pd.read_excel(PATH_CRECIMIENTO + "EV data by country 2026.xlsx", sheet_name="GEVO_EV_2026")

In [ ]:
def trimestral(cuadro):
    """Filas trimestrales de un cuadro del IDECBA, con el año de la fila de cabecera que las precede."""
    periodo = cuadro["periodo"].astype(str)
    anio = periodo.str.extract(r"^(\d{4})", expand=False).astype(float).ffill()
    trimestre = periodo.str.extract(r"^([1-4])\w*\.\s*trimestre", expand=False)     # "1er. trimestre*", "3er.trimestre"
    return (cuadro.assign(anio=anio, trimestre=trimestre).dropna(subset=["trimestre"])
            .astype({"anio": int, "trimestre": int}).reset_index(drop=True))

def anual(cuadro, columna):
    """Valor de la fila de cabecera de cada año (el total anual), indexado por año; los años sin total se omiten."""
    filas = cuadro[cuadro["periodo"].astype(str).str.fullmatch(r"\d{4}(\.0)?\*?")]
    return filas.set_index(filas["periodo"].astype(str).str[:4].astype(int))[columna].dropna().astype(float)

patentamientos_q, distribucion_q = trimestral(patentamientos), trimestral(distribucion)

# Controles de la lectura: los trimestres suman el total anual de autos, y el % de electrificados de cada año es el
# promedio de sus trimestres ponderado por el total de automotores del otro cuadro, lo que confirma que los dos
# cuadros quedan alineados. Entre cuadros hay diferencias de hasta 0,005 puntos (2021) por revisiones de los datos;
# un trimestre desalineado daría diferencias de puntos enteros.
autos_anuales = anual(patentamientos, "autos")
assert np.allclose(patentamientos_q.groupby("anio")["autos"].sum().loc[autos_anuales.index], autos_anuales), \
    "Los trimestres no suman el total anual de autos."
conjunto = distribucion_q.merge(patentamientos_q, on=["anio", "trimestre"], validate="one_to_one")
pct_ponderado = ((conjunto["pct_electrificados"] * conjunto["total"]).groupby(conjunto["anio"]).sum()
                 / conjunto.groupby("anio")["total"].sum())
pct_anual = anual(distribucion, "pct_electrificados")
assert np.allclose(pct_ponderado.loc[pct_anual.index], pct_anual, rtol=0, atol=0.01), \
    "El % anual de electrificados no cierra con los trimestres."

caba = (patentamientos_q[patentamientos_q["anio"] >= distribucion_q["anio"].min()]
        .merge(distribucion_q[["anio", "trimestre", "pct_electrificados"]], on=["anio", "trimestre"], how="left",
               validate="one_to_one"))
assert caba["pct_electrificados"].notna().all() and len(caba) == len(distribucion_q), "Los cuadros no cubren los mismos trimestres."
print(f"CABA: {len(caba)} trimestres, del {caba['trimestre'].iloc[0]}.º de {caba['anio'].iloc[0]} "
      f"al {caba['trimestre'].iloc[-1]}.º de {caba['anio'].iloc[-1]}. Controles de lectura: OK "
      f"({len(autos_anuales)} totales anuales de autos, {len(pct_anual)} de % de electrificados)")

**Magnitud ajustada.** Se ajusta el **stock de autos enchufables (BEV + PHEV) que cargan en estaciones públicas** de CABA, en vehículos: es la demanda potencial de carga, con la que escala el IA del motor. Se obtiene acumulando, trimestre a trimestre, los enchufables patentados que cargan en estaciones:

`autos nuevos × % de electrificados × % de enchufables entre los electrificados × % que carga en estaciones`

Los electrificados son la categoría «Ecológico» del IDECBA (eléctricos e híbridos, enchufables o no).

**Supuestos adoptados:**
1. *Escala temporal.* Cada stock se ubica al cierre de su trimestre y cada participación en las ventas, en el punto medio. Para los países (datos anuales), el stock va al cierre del año y la participación en ventas, a mitad de año.
2. *Autos.* Se toman los patentamientos de automóviles, sin comerciales livianos ni pesados. El IDECBA publica la distribución por tipo de motor para el total de automotores; se la aplica a los automóviles.
3. *Composición del mercado.* La proporción de enchufables entre los electrificados patentados en CABA es la nacional del mismo año (SIOMAA), porque no hay un dato propio de CABA. Se aplica al flujo de patentamientos de cada trimestre.
4. *Stock inicial.* El stock de enchufables anterior a 2020 se toma nulo: los informes del SIOMAA registran 0 % de enchufables entre los electrificados patentados hasta 2019.
5. *Sin bajas.* No se descuentan bajas del parque: el enchufable más antiguo de la serie tiene seis años y medio.
6. *Carga pública.* El 29 % de los usuarios carga en estaciones y el resto en su domicilio (valor de México en IEA, *Global EV Outlook 2024*), en proporción constante en el tiempo.

In [ ]:
# Proporción de enchufables (BEV + PHEV) entre los electrificados patentados en Argentina, en %, por año.
# Fuente: informes de electromovilidad del SIOMAA (2026: primer semestre).
PCT_ENCHUFABLES_ANIO = {2020: 1.0, 2021: 1.0, 2022: 2.5, 2023: 3.8, 2024: 11.0, 2025: 21.0, 2026: 30.0}
# Fracción de los usuarios que carga en estaciones públicas; el resto carga en su domicilio (IEA, GEVO 2024, México).
PORC_CARGA_ESTACIONES = 0.29

caba["anio_stock"] = caba["anio"] + caba["trimestre"] / 4               # cierre del trimestre
caba["anio_ventas"] = caba["anio"] + (caba["trimestre"] - 0.5) / 4      # punto medio del trimestre
caba["pct_enchufables"] = caba["anio"].map(PCT_ENCHUFABLES_ANIO)
caba["pct_enchufables_ventas"] = caba["pct_electrificados"] * caba["pct_enchufables"] / 100   # % de los autos nuevos
caba["enchufables_nuevos"] = caba["autos"] * caba["pct_enchufables_ventas"] / 100
caba["stock"] = caba["enchufables_nuevos"].cumsum() * PORC_CARGA_ESTACIONES
assert caba["pct_enchufables"].notna().all(), "Falta la proporción de enchufables de algún año."

display(caba[["anio", "trimestre", "autos", "pct_electrificados", "pct_enchufables", "pct_enchufables_ventas",
              "stock"]].round(3))

### 2.2. Modelos de crecimiento

**Idea original: la logística.** La propuesta plantea la curva logística `P(t) = K / (1 + A·e^(−r·t))`, la forma clásica de una adopción en "S". Al proyectarla para CABA, el resultado no fue realista. La logística es simétrica respecto de su inflexión, que ocurre a la mitad del techo (`P = K/2`): frena al mismo ritmo al que aceleró. Como la serie de CABA está en plena aceleración, el ajuste le asigna la velocidad alta del tramo reciente y, por la simetría, la obliga a frenar igual de rápido. La curva resultante recorre toda la "S" en unos cuatro años: para sostener ese ritmo, hacia fines de 2028 el 90 % de los autos nuevos tendría que ser enchufable, una proporción que seis de los siete mercados maduros de 2.3 no alcanzan ni en su techo, y a partir de 2033 la curva deja de sumar enchufables, como si en CABA casi no se patentaran más. Los números están en 2.4.

**Alternativas.** Se probaron tres curvas más flexibles:
- **Gompertz**, `P(t) = K·e^(−b·e^(−c·t))`: asimétrica, con la inflexión en `P = K/e` (37 % del techo). Crece rápido al principio y se aproxima lentamente a la saturación. En Pham (2023) es el mejor modelo para varias de las series nórdicas.
- **Richards** (logística generalizada): agrega a la logística un parámetro de forma. La serie de CABA no alcanza para determinarlo y el ajuste elige formas extremas, que aceleran hasta el final y frenan de golpe, con cuotas implícitas de enchufables por encima del 100 % de los patentamientos. Se descarta.
- **Bass**: modelo de difusión con coeficientes de innovación e imitación. Anclado al stock observado de CABA, el coeficiente de innovación queda prácticamente nulo y la curva coincide con la logística, así que no aporta nada distinto. Se descarta.

La comparación de 2.4 queda entre la logística y Gompertz. En las dos, `t` se mide en años desde el origen de cada serie (para CABA, desde el 1/1/2020).

In [ ]:
def modelo_logistico(t, K, A, r):
    """Logística: P(t) = K / (1 + A·e^(−r·t)). Simétrica: inflexión en t = ln(A)/r, con P = K/2."""
    return K / (1 + A * np.exp(-r * t))

def modelo_gompertz(t, K, b, c):
    """Gompertz: P(t) = K·exp(−b·e^(−c·t)). Asimétrica: inflexión en t = ln(b)/c, con P = K/e."""
    return K * np.exp(-b * np.exp(-c * t))

MODELOS = {"Logística": modelo_logistico, "Gompertz": modelo_gompertz}
COTA_K = 1000   # K se acota a 1000 veces el último dato: un K que queda en la cota no es identificable con esa serie

def ajustar(nombre, t, y):
    """Mínimos cuadrados en niveles con varios arranques: K como múltiplo del último dato, dos velocidades y la
    posición que hace pasar la curva por el primer dato. Devuelve (parámetros, covarianza) del de menor error."""
    funcion, inf, sup = MODELOS[nombre], [0, 0, 0], [COTA_K * y[-1], np.inf, 10]
    mejor = None
    for r in (0.3, 1.0):
        for k in (1.5, 5, 50):
            K = k * y[-1]
            posicion = K / y[0] - 1 if nombre == "Logística" else np.log(K / y[0])
            p0 = np.clip([K, posicion * np.exp(r * t[0]), r], inf, sup)
            with warnings.catch_warnings(), np.errstate(over="ignore", invalid="ignore", divide="ignore"):
                warnings.simplefilter("ignore", OptimizeWarning)
                try:
                    params, cov = curve_fit(funcion, t, y, p0=p0, bounds=(inf, sup), x_scale="jac", maxfev=2000)
                except RuntimeError:      # el arranque no convergió: se prueba el siguiente
                    continue
            sse = np.sum((y - funcion(t, *params)) ** 2)
            if np.isfinite(sse) and (mejor is None or sse < mejor[2]):
                mejor = (params, cov, sse)
    if mejor is None:
        raise RuntimeError(f"{nombre}: ningún arranque convergió.")
    return mejor[0], mejor[1]

### 2.3. Techo del parque: analogía con los mercados maduros

La serie de CABA está en plena aceleración y todavía no muestra su inflexión, así que por sí sola no permite estimar el techo `K`. Se lo toma de países que ya atravesaron la etapa en la que hoy está CABA.

**Indicador de etapa.** Es la participación de los enchufables en las ventas de autos nuevos, la magnitud disponible tanto para CABA como para los países: CABA está en 9,1 % (segundo trimestre de 2026). Como referencia se usan los siete mercados maduros de la base de la IEA, que ya pasaron la inflexión de esa cuota y por eso muestran su techo: **Noruega, Suecia, Finlandia, Islandia, Dinamarca, Países Bajos y China**. Los países latinoamericanos de la base están en la misma etapa que CABA o detrás, así que todavía no muestran ni su inflexión ni su techo.

**Techo de CABA.** Para cada país se ubica el instante en que su cuota de enchufables en las ventas cruzó el 9,1 % actual de CABA, y se calcula cuántas veces creció su stock desde ese cruce hasta su techo, `K / P(cruce)`. El techo de CABA es `K = P(T=0) × mediana de K / P(cruce)`, con `T = 0` del motor en el 1/7/2026, cierre del último trimestre observado. Así, `K` no depende del parque total de CABA ni del nivel futuro de los patentamientos.

El techo de cada país se estima con una logística ajustada a su stock: como esos países ya pasaron la inflexión, el techo es observable y la logística lo determina con poco error. Gompertz reproduce bien la forma de esas series, pero por su aproximación lenta extrapola el techo muy por encima del último dato y lo deja indeterminado en varias de ellas (columnas de error de la tabla). De los mismos países sale el **techo de la cuota** de enchufables en las ventas, `S_MAX`, asíntota de una logística ajustada a su cuota anual, que se usa como control en 2.4.

In [ ]:
PAISES_MADUROS = ["Noruega", "Suecia", "Finlandia", "Islandia", "Dinamarca", "Países Bajos", "China"]
NOMBRES_PAISES = {"Norway": "Noruega", "Sweden": "Suecia", "Finland": "Finlandia", "Iceland": "Islandia",
                  "Denmark": "Dinamarca", "Netherlands": "Países Bajos", "China": "China"}
MAGNITUDES_IEA = {"EV stock": "stock", "EV sales": "ventas", "EV sales share": "pct_ventas"}

# IEA: autos (powertrain "EV" = BEV + PHEV), datos históricos; stock y ventas en vehículos, cuota en %.
paises = (iea[(iea["category"] == "Historical") & (iea["mode"] == "Cars") & (iea["powertrain"] == "EV")
              & iea["parameter"].isin(list(MAGNITUDES_IEA)) & iea["region_country"].isin(list(NOMBRES_PAISES))]
          .assign(pais=lambda d: d["region_country"].map(NOMBRES_PAISES),
                  magnitud=lambda d: d["parameter"].map(MAGNITUDES_IEA))
          .pivot_table(index=["pais", "year"], columns="magnitud", values="value")
          .reset_index().rename(columns={"year": "anio"}))

T_INICIO = 6.5                                                    # T = 0 del motor: 1/7/2026, en años desde el 1/1/2020
P_INICIO = caba["stock"].iloc[-1]                                 # stock observado en T = 0
PCT_ENCHUFABLES_CABA = caba["pct_enchufables_ventas"].iloc[-1]    # cuota en las ventas del 2.º trimestre de 2026

# Paleta categórica en orden fijo; CABA se destaca en negro en todos los gráficos.
PALETA = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
COLOR_CABA = "#0b0b0b"

def formato_ejes(ax, titulo, xlabel, ylabel):
    ax.set_title(titulo, fontsize=11)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(color="#e5e5e5", linewidth=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.legend(frameon=False, fontsize=9)

fig, ax = plt.subplots(figsize=(10, 5))
for i, pais in enumerate(PAISES_MADUROS):
    datos = paises[(paises["pais"] == pais) & (paises["anio"] >= 2010)]
    ax.plot(datos["anio"] + 0.5, datos["pct_ventas"], marker="o", markersize=3, color=PALETA[i], label=pais)
ax.plot(caba["anio_ventas"], caba["pct_enchufables_ventas"], color=COLOR_CABA, linewidth=2.2, label="CABA (trimestral)")
ax.axhline(PCT_ENCHUFABLES_CABA, color=COLOR_CABA, linewidth=1, linestyle=":",
           label=f"Cuota actual de CABA ({PCT_ENCHUFABLES_CABA:.1f} %)")
formato_ejes(ax, "Enchufables en las ventas de autos nuevos: CABA y mercados maduros", "Año", "% de las ventas")
plt.tight_layout()
plt.show()

In [ ]:
def anio_cruce(anios, valores, umbral):
    """Primer instante en que la serie alcanza el umbral (interpolación log-lineal entre observaciones); NaN si no lo alcanza."""
    anios, valores = np.asarray(anios, float), np.asarray(valores, float)
    for i in range(1, len(valores)):
        if valores[i - 1] < umbral <= valores[i] and valores[i - 1] > 0:
            frac = (np.log(umbral) - np.log(valores[i - 1])) / (np.log(valores[i]) - np.log(valores[i - 1]))
            return anios[i - 1] + frac * (anios[i] - anios[i - 1])
    return np.nan

def logistica_cuota(t, S_MAX, t_inf, r):
    """Cuota de enchufables en las ventas (%): logística con techo S_MAX, inflexión en t_inf y velocidad r."""
    return S_MAX / (1 + np.exp(-r * (t - t_inf)))

def techo_cuota(t, y):
    """S_MAX por mínimos cuadrados con varios arranques, acotado a 100 %; t en años desde 2000."""
    mejor = None
    for S0 in (y.max() * 1.1, 60, 100):
        for r0 in (0.3, 0.7, 1.2):
            try:
                params, _ = curve_fit(logistica_cuota, t, y, p0=[min(S0, 100), t[np.argmax(y > y.max() / 2)], r0],
                                      bounds=([0, 0, 0], [100, 60, 5]), maxfev=20000)
            except RuntimeError:      # el arranque no convergió: se prueba el siguiente
                continue
            sse = np.sum((y - logistica_cuota(t, *params)) ** 2)
            if mejor is None or sse < mejor[1]:
                mejor = (params, sse)
    return mejor[0][0]

# Son 7 países con tres ajustes chicos cada uno: se corren en serie (levantar procesos cuesta más que el ajuste).
filas = []
for pais in PAISES_MADUROS:
    datos = paises[paises["pais"] == pais]
    # Cruce: cuota a mitad de año; stock al cierre del año y ventas anuales, interpolados al instante del cruce.
    con_cuota = datos.dropna(subset=["pct_ventas", "stock"])
    cruce = anio_cruce(con_cuota["anio"] + 0.5, con_cuota["pct_ventas"], PCT_ENCHUFABLES_CABA)
    stock_cruce = np.exp(np.interp(cruce, con_cuota["anio"] + 1, np.log(con_cuota["stock"])))
    ventas_cruce = np.exp(np.interp(cruce, con_cuota["anio"] + 0.5, np.log(con_cuota["ventas"])))
    # Techo del stock: t en años desde el inicio de la adopción registrada (P = 0 en t = 0).
    con_stock = datos[datos["stock"] > 0]
    origen = con_stock["anio"].min()
    t, y = (con_stock["anio"] + 1 - origen).to_numpy(float), con_stock["stock"].to_numpy(float)
    ajustes = {nombre: ajustar(nombre, t, y) for nombre in MODELOS}
    (K, A, r), cov = ajustes["Logística"]
    assert not np.isclose(K, COTA_K * y[-1]), f"{pais}: el techo de la logística quedó en la cota."
    error_K = {nombre: 100 * np.sqrt(cov_m[0, 0]) / params_m[0] for nombre, (params_m, cov_m) in ajustes.items()}
    con_cuota_pos = datos[datos["pct_ventas"] > 0]
    filas.append({"país": pais, "cruce (año)": cruce, "stock / ventas en el cruce (años)": stock_cruce / ventas_cruce,
                  "error est. de K, logística (%)": error_K["Logística"],
                  "error est. de K, Gompertz (%)": error_K["Gompertz"],
                  "K / P(cruce)": K / modelo_logistico(cruce - origen, K, A, r),
                  "S_MAX (%)": techo_cuota((con_cuota_pos["anio"] + 0.5 - 2000).to_numpy(float),
                                           con_cuota_pos["pct_ventas"].to_numpy(float))})

referencias = pd.DataFrame(filas).set_index("país")
referencias.loc["Mediana", ["K / P(cruce)", "S_MAX (%)"]] = referencias[["K / P(cruce)", "S_MAX (%)"]].median()
display(referencias.round(2))

cociente = referencias.loc[PAISES_MADUROS, "K / P(cruce)"]
K_CABA = P_INICIO * cociente.median()
S_MAX_MADUROS = referencias.loc["Mediana", "S_MAX (%)"]
# CABA: stock de enchufables (sin descontar la carga domiciliaria) sobre los enchufables patentados en el último año.
stock_sobre_ventas_caba = caba["enchufables_nuevos"].sum() / caba["enchufables_nuevos"].iloc[-4:].sum()
print(f"K / P(cruce): mediana {cociente.median():.2f} | mínimo {cociente.min():.2f} ({cociente.idxmin()}) | "
      f"máximo {cociente.max():.2f} ({cociente.idxmax()})")
print(f"Techo de CABA: K = {P_INICIO:,.0f} × {cociente.median():.2f} = {K_CABA:,.0f} vehículos")
print(f"Stock / ventas anuales de enchufables de CABA en T = 0: {stock_sobre_ventas_caba:.2f} años | "
      f"techo de la cuota de los mercados maduros (mediana): {S_MAX_MADUROS:.1f} %")

- `K / P(cruce)` va de 14,8 (Suecia) a 57,4 (Países Bajos), con mediana 33,3 (Dinamarca). El techo de CABA es de 56 294 vehículos, 33 veces el stock de `T = 0`.
- En el cruce, el stock de los países equivalía a entre 0,85 y 2,6 años de ventas de enchufables; el de CABA equivale a 1,3, dentro de ese rango. La profundidad del parque en el momento del cruce es comparable, lo que respalda la analogía.
- La logística determina el techo de los siete países con un error estándar de entre 1,8 % y 24 %; Gompertz, con errores de hasta 85 % (China) y 316 % (Países Bajos).
- El techo de la cuota de enchufables en las ventas va de 57 % (Finlandia) a 99 % (Noruega), con mediana 63,7 %.

### 2.4. Comparación de la logística y Gompertz

Las dos curvas usan el techo `K` de 2.3 y se anclan al stock observado en `T = 0`, lo que fija su parámetro de posición (`A` en la logística, `b` en Gompertz). Queda un solo parámetro libre, la velocidad (`r` en la logística, `c` en Gompertz), que se ajusta por mínimos cuadrados en niveles sobre la serie de CABA.

En el tramo observado las dos curvas aceleran; se diferencian en cómo frenan, que es lo que importa para proyectar. Por eso se las compara por lo que proyectan en una ventana de 10 años desde `T = 0`, que debe cubrir el horizonte `TF` de la simulación:
- **Crecimiento** del stock a 1, 3, 4, 5 y 10 años: es la inversa del factor que multiplicará al IA. Se informan también la inflexión y el año en que la curva llega al 95 % del techo.
- **Cuota implícita**: el % de los autos nuevos que tendría que ser enchufable para producir la pendiente de la curva, `dP/dt / (ventas anuales × 29 %)`. En `T = 0` tiene que parecerse al 9,1 % observado. Nunca puede superar el 100 % (**cota física**: el stock no puede crecer más rápido que si todos los autos nuevos fueran enchufables), y no debería superar el techo de la cuota de los mercados maduros (2.3).

Las ventas anuales de autos nuevos se mantienen en el nivel de los últimos cuatro trimestres. Como ese supuesto no entra en la curva, la cuota máxima se informa también con el nivel de 2023–2024 (unos 55 000 patentamientos por año).

In [ ]:
HORIZONTE_ANIOS = 10    # ventana de proyección: supuesto, debe cubrir el horizonte TF de la simulación
VENTAS_ANUALES_CABA = caba["autos"].iloc[-4:].sum()                 # últimos cuatro trimestres
COTA_CRECIMIENTO = VENTAS_ANUALES_CABA * PORC_CARGA_ESTACIONES     # vehículos/año con el 100 % de los autos nuevos enchufables
COTA_CRECIMIENTO_23_24 = caba.loc[caba["anio"].isin([2023, 2024]), "autos"].sum() / 2 * PORC_CARGA_ESTACIONES

def logistica_anclada(t, K, r):
    """Logística que pasa por P_INICIO en T_INICIO: A queda determinado por K y r."""
    return modelo_logistico(t, K, (K / P_INICIO - 1) * np.exp(r * T_INICIO), r)

def gompertz_anclada(t, K, c):
    """Gompertz que pasa por P_INICIO en T_INICIO: b queda determinado por K y c."""
    return modelo_gompertz(t, K, np.log(K / P_INICIO) * np.exp(c * T_INICIO), c)

CURVAS = {  # nombre: (curva anclada, arranques de la velocidad)
    "Logística": (logistica_anclada, [0.3, 1.0, 2.0]),
    "Gompertz": (gompertz_anclada, [0.1, 0.3, 1.0]),
}
COTA_VELOCIDAD = 10   # 1/año
t_caba, y_caba = (caba["anio_stock"] - 2020).to_numpy(), caba["stock"].to_numpy()

def ajustar_velocidad(curva, K):
    """Velocidad por mínimos cuadrados en niveles, con K fijo y varios arranques; devuelve la de menor error."""
    funcion, arranques = CURVAS[curva]
    mejor = None
    for v0 in arranques:
        with warnings.catch_warnings(), np.errstate(over="ignore", invalid="ignore", divide="ignore"):
            warnings.simplefilter("ignore", OptimizeWarning)
            try:
                (v,), _ = curve_fit(lambda t, v: funcion(t, K, v), t_caba, y_caba, p0=[v0],
                                    bounds=([0], [COTA_VELOCIDAD]), maxfev=5000)
            except RuntimeError:      # el arranque no convergió: se prueba el siguiente
                continue
            sse = np.sum((y_caba - funcion(t_caba, K, v)) ** 2)
        if np.isfinite(sse) and (mejor is None or sse < mejor[1]):
            mejor = (v, sse)
    if mejor is None:
        raise RuntimeError(f"{curva}: ningún arranque convergió.")
    return mejor[0]

T_PROY = np.linspace(T_INICIO, T_INICIO + HORIZONTE_ANIOS, 1201)
T_LARGO = np.linspace(0, 60, 6001)      # inflexión y 95 % de K, aunque caigan fuera de la ventana
ANIOS_PROY = (1, 3, 4, 5, 10)
COLUMNAS_CRECIMIENTO = [f"× {a} año{'s' if a > 1 else ''}" for a in ANIOS_PROY]

filas, curvas, velocidades = [], {}, {}
for curva, (funcion, _) in CURVAS.items():
    v = ajustar_velocidad(curva, K_CABA)
    assert not (np.isclose(v, 0) or np.isclose(v, COTA_VELOCIDAD)), f"{curva}: la velocidad quedó en la cota."
    P = lambda t, f=funcion, v=v: f(t, K_CABA, v)
    curvas[curva], velocidades[curva] = P, v
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        p_proy, p_largo = P(T_PROY), P(T_LARGO)
    pendiente = np.gradient(p_proy, T_PROY)
    llega_95 = p_largo >= 0.95 * K_CABA
    filas.append({
        "curva": curva,
        **{columna: P(T_INICIO + a) / P_INICIO for columna, a in zip(COLUMNAS_CRECIMIENTO, ANIOS_PROY)},
        "inflexión (año)": 2020 + T_LARGO[np.argmax(np.gradient(p_largo, T_LARGO))],
        "95 % de K (año)": 2020 + T_LARGO[np.argmax(llega_95)] if llega_95.any() else np.nan,
        "cuota en T=0": 100 * pendiente[0] / COTA_CRECIMIENTO,
        "cuota máx.": 100 * pendiente.max() / COTA_CRECIMIENTO,
        "cuota máx. (mercado 2023–2024)": 100 * pendiente.max() / COTA_CRECIMIENTO_23_24,
        "cuota a 10 años": 100 * pendiente[-1] / COTA_CRECIMIENTO,
    })

proyeccion = pd.DataFrame(filas).set_index("curva")
print(f"K = {K_CABA:,.0f} vehículos | velocidad ajustada: logística r = {velocidades['Logística']:.3f}/año, "
      f"Gompertz c = {velocidades['Gompertz']:.3f}/año")
print("Proyección desde T = 0 (stock relativo al de T = 0) y controles. Cuotas: % de enchufables en los patentamientos "
      f"implícito en la curva; observado en T = 0: {PCT_ENCHUFABLES_CABA:.1f} %; techo de la cuota en los mercados "
      f"maduros: {S_MAX_MADUROS:.1f} %.")
display(proyeccion.round(1))

In [ ]:
# Stock proyectado y cuota implícita de las dos curvas.
COLOR_CURVA = {curva: PALETA[i] for i, curva in enumerate(CURVAS)}
t_graf = np.linspace(t_caba[0], T_INICIO + HORIZONTE_ANIOS, 500)

fig, ejes = plt.subplots(1, 2, figsize=(15, 5.5))
for curva, P in curvas.items():
    estilo = dict(color=COLOR_CURVA[curva], linewidth=1.8, label=curva)
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        ejes[0].plot(2020 + t_graf, P(t_graf), **estilo)
        ejes[1].plot(2020 + T_PROY, 100 * np.gradient(P(T_PROY), T_PROY) / COTA_CRECIMIENTO, **estilo)
ejes[0].plot(caba["anio_stock"], caba["stock"], "o", color=COLOR_CABA, markersize=3, label="CABA (observado)")
ejes[0].set_ylim(bottom=0)
formato_ejes(ejes[0], "Stock de enchufables que cargan en estaciones (CABA)", "Año", "Vehículos")
ejes[1].plot(caba["anio_ventas"], caba["pct_enchufables_ventas"], "o", color=COLOR_CABA, markersize=3,
             label="CABA (observado)")
ejes[1].axhline(100, color=COLOR_CABA, linewidth=1, linestyle=":", label="Cota física")
ejes[1].axhline(S_MAX_MADUROS, color=COLOR_CABA, linewidth=1, linestyle="-.", label="Techo de la cuota (maduros)")
formato_ejes(ejes[1], "% de enchufables en los patentamientos implícito en cada curva", "Año", "% de los patentamientos")
for ax in ejes:
    ax.axvline(2020 + T_INICIO, color="#999999", linewidth=0.8)
plt.tight_layout()
plt.show()

**Análisis comparativo.**
- Las dos curvas reproducen el ritmo actual de CABA: implican que en `T = 0` es enchufable el 10,5 % (logística) y el 9,0 % (Gompertz) de los autos nuevos, contra el 9,1 % observado.
- **Logística.** La velocidad que toma del tramo reciente, sumada a su simetría, comprime toda la "S" en cuatro años: el stock se multiplica por 25 a los 3 años, llega al 95 % del techo en el segundo semestre de 2030 y desde ahí queda prácticamente plano. Para crecer así, hacia fines de 2028 el 90 % de los autos nuevos tendría que ser enchufable, muy por encima del techo de la cuota de los mercados maduros (mediana de 63,7 %); con el nivel de patentamientos de 2023–2024, el 134 %, por encima de la cota física. A partir de 2033 la cuota implícita cae por debajo del 1 %: la curva supone que en CABA casi no se patentan más enchufables.
- **Gompertz.** Crece de forma gradual: ×2,9, ×10,2, ×18,7 y ×30,3 a 1, 3, 5 y 10 años. Su cuota implícita máxima es de 31 %, en 2030 (47 % con el mercado de 2023–2024), siempre por debajo del techo de la cuota de los mercados maduros, y a los 10 años todavía suma enchufables (7 % de los patentamientos). Llega al 95 % del techo en 2038, fuera de la ventana de proyección.
- La logística ajusta mejor el tramo observado, pero ahí las dos curvas aceleran y sólo se diferencian en cómo frenan: ajustar mejor el pasado no la favorece para proyectar.

**Conclusión.** Gompertz es la proyección más realista: reproduce el ritmo actual de CABA y respeta la cota física y el techo de la cuota de los mercados maduros en todo el horizonte. Se adopta **Gompertz con el techo de la mediana de los siete mercados maduros**, lo que corrige la forma de la curva planteada en la propuesta.

### 2.5. Factor de crecimiento aplicado al Intervalo entre Arribos

Función que, dado el instante `T` de la simulación (minutos desde el inicio), devuelve el factor que multiplica el `IA` generado. El factor se calcula sobre el stock, `P(T=0) / P(T)`, porque los arribos a las estaciones los generan todos los autos enchufables en circulación, no sólo los que se incorporan en ese momento. Vale 1 en `T = 0` y decrece a medida que crece el parque: el IA se achica rápido durante la aceleración y cada vez más lento en la desaceleración, hasta estabilizarse en `P(T=0) / K` cuando el parque se satura.

In [ ]:
# Curva adoptada: Gompertz con el techo de la mediana de los mercados maduros (2.4).
# PARAMS_CRECIMIENTO = (K, b, c): K en vehículos, c en 1/año, t en años desde el 1/1/2020.
C_CRECIMIENTO = velocidades["Gompertz"]
PARAMS_CRECIMIENTO = (K_CABA, np.log(K_CABA / P_INICIO) * np.exp(C_CRECIMIENTO * T_INICIO), C_CRECIMIENTO)
MIN_POR_ANIO = 12 * 30 * 24 * 60    # 518 400 min: año de 12 meses de 30 días, la misma convención que TCE

def P_crecimiento(t, params=PARAMS_CRECIMIENTO):
    """Stock de enchufables que cargan en estaciones en CABA (Gompertz); t en años desde el 1/1/2020."""
    K, b, c = params
    return K * math.exp(-b * math.exp(-c * t))     # K·exp(−b·e^(−c·t))

def factor_crecimiento(T, params=PARAMS_CRECIMIENTO):
    """Factor que multiplica el IA generado en el instante T (min desde el inicio de la simulación).
    Es P(T = 0) / P(T): vale 1 al inicio y decrece hacia P(T = 0) / K a medida que el parque satura."""
    return P_crecimiento(T_INICIO, params) / P_crecimiento(T_INICIO + T / MIN_POR_ANIO, params)

print("Gompertz adoptada: K = {:,.0f} vehículos | b = {:.4f} | c = {:.4f}/año".format(*PARAMS_CRECIMIENTO))

# Chequeos sobre el factor (valores, no ausencia de excepciones).
T_grilla = np.linspace(0, HORIZONTE_ANIOS * MIN_POR_ANIO, 500)
factor_grilla = np.array([factor_crecimiento(T) for T in T_grilla])
limite = P_INICIO / PARAMS_CRECIMIENTO[0]
assert math.isclose(P_crecimiento(T_INICIO), P_INICIO), "La curva no pasa por el stock observado en T = 0."
assert all(math.isclose(P_crecimiento(T_INICIO + a), curvas["Gompertz"](T_INICIO + a)) for a in ANIOS_PROY), \
    "La curva adoptada no coincide con la Gompertz de 2.4."
assert math.isclose(factor_grilla[0], 1.0), "El factor no vale 1 en T = 0."
assert np.all(np.diff(factor_grilla) < 0), "El factor no es estrictamente decreciente."
assert math.isclose(factor_crecimiento(1e12), limite, rel_tol=1e-9), "El factor no tiende a P(T=0)/K."
assert factor_grilla[-1] > limite, "El factor cruzó su límite inferior."

display(pd.DataFrame([{"T (años)": a, "factor sobre el IA": factor_crecimiento(a * MIN_POR_ANIO),
                       "flujo de arribos relativo a T = 0": 1 / factor_crecimiento(a * MIN_POR_ANIO)}
                      for a in (1, 3, 5, 10)]).set_index("T (años)").round(3))

In [ ]:
# Factor sobre el IA en la ventana de proyección.
anios_grilla = 2020 + T_INICIO + T_grilla / MIN_POR_ANIO
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(anios_grilla, factor_grilla, color=PALETA[0], linewidth=1.8)
ax.axhline(limite, color=COLOR_CABA, linewidth=1, linestyle=":", label=f"Límite al saturar, P(T=0) / K = {limite:.3f}")
ax.set_yscale("log")
formato_ejes(ax, "Factor de crecimiento sobre el IA (Gompertz)", "Año", "Factor sobre el IA (escala logarítmica)")
plt.tight_layout()
plt.show()

**Factor adoptado** (Gompertz): `factor_crecimiento(T)` vale 0,345 a 1 año, 0,098 a 3, 0,053 a 5 y 0,033 a 10. Es decir, el flujo de arribos de cada estación se multiplica por 2,9, 10,2, 18,7 y 30,3 respecto de `T = 0`. La función es escalar (`math.exp`), está definida a nivel de módulo para que sea serializable en los procesos del barrido, y recibe los parámetros como argumento.

`T = 0` supone que las FDPs de IA ajustadas en la sección 5 representan la demanda de una estación al 1/7/2026. Ese supuesto se explicita también al ajustarlas.

## 3. Carga de Datos Inicial

En esta sección se carga el dataset para la generación de las Funciones de Densidad de Probabilidad (FDP)

> **[DESACTUALIZADO — TP 4]** La ruta y el archivo son los del TP 4. El TP Final usa `Dataset California.csv` en `/content/drive/MyDrive/Colab Notebooks/TP Final Simu/Datos/`.

In [ ]:
# [DESACTUALIZADO — TP 4] actualizar PATH_DATOS y el nombre del dataset.
# Ruta de acceso a los datos
PATH_DATOS = "/content/drive/MyDrive/Colab Notebooks/TP 4 Simu/Datos/"

try:
    # Carga de datasets para el análisis de variables estocásticas
    df = pd.read_csv(PATH_DATOS + "EVChargingStationUsage.csv")

    print("Dataset inicializado correctamente.")
except FileNotFoundError as e:
    print(f"Error en la carga de recursos: {e}")

## 4. Obtención de la FDP del Tiempo de Carga (TC)

El Tiempo de Carga representa la duración de la sesión de carga. Se utiliza la columna `Charging Time (hh:mm:ss)` del dataset mapeada a minutos para determinar la distribución estocástica que mejor represente este proceso, y poder generar valores aleatorios para dicha distribución.

> **[DESACTUALIZADO — TP 4]** Reajustar la FDP de `TC` sobre el dataset del TP Final. De este ajuste salen `TCP` (`E[TC]` truncado) y `FI = 41,7 / TCP`; no se reciclan los valores del TP 4.

In [ ]:
# [DESACTUALIZADO — TP 4] reejecutar con el dataset del TP Final.
def mostrar_histograma(data, title, xlabel):
    plt.figure(figsize=(7,5))
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Densidad de Probabilidad")
    plt.hist(data, bins=100, density=True)
    plt.show()

df["Charging Time (minutes)"] = (
    pd.to_timedelta(df["Charging Time (hh:mm:ss)"].dropna())
    .dt.total_seconds() / 60
)

mostrar_histograma(df["Charging Time (minutes)"], "Histograma del Tiempo de Carga", "Tiempo de Carga (minutos)")

Dado que la distribución tiene una cola derecha larga, se verá las distribuciónes que mejor se ajusten usando como criterio el `ks_statistic`.

In [ ]:
# [DESACTUALIZADO — TP 4] reejecutar con el dataset del TP Final.
def mostrar_mejores_dist(data, nombre, criterio):
    """Ajusta y visualiza las mejores distribuciones para una serie de datos."""
    print(f"--- Ajustando distribuciones para {nombre} según {criterio} ---")
    f = Fitter(data)
    f.fit()
    display(f.summary(method=criterio))
    return f

f_tc = mostrar_mejores_dist(df["Charging Time (minutes)"], "Tiempo de Carga", "ks_statistic")

Se selecciona la distribución `Gumbel R`, pues tiene el ks_statistic más bajo, y también gana en sumsquare_error y kl_div.

In [ ]:
# [DESACTUALIZADO — TP 4] reejecutar con el dataset del TP Final.
def mostrar_fdp(title, xlabel, f, fdp, data):
    plt.figure(figsize=(7,5))
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Densidad de Probabilidad")
    plt.hist(data, bins=100, density=True, alpha=0.5)
    f.plot_pdf(names=[fdp])
    plt.show()

def obtener_dist(f, funcion, nombre, data):
    """Extrae una distribución scipy lista para usar a partir de un Fitter ajustado."""
    if funcion not in f.fitted_param:
        disponibles = list(f.fitted_param.keys())
        raise ValueError(f"'{funcion}' no fue ajustada. Disponibles: {disponibles}")

    params_tuple = f.fitted_param[funcion]
    dist_class   = getattr(stats, funcion)
    distribucion = dist_class(*params_tuple)

    param_names  = (dist_class.shapes.split(", ") if dist_class.shapes else []) + ["loc", "scale"]
    params_dict  = dict(zip(param_names, params_tuple))

    print(f"La función elegida para {nombre} es: {funcion}")
    print(f"Con los parámetros: {params_dict}")
    print(f"Acotada entre: {data.min()} y {data.max()}")

    mostrar_fdp(f"FDP para {nombre}", f"{nombre} (minutos)", f, funcion, data)

    return distribucion

dist_tc = obtener_dist(f_tc, "gumbel_r", "Tiempo de Carga", df["Charging Time (minutes)"])

def rvs_truncado(dist, data):
    """
    Genera muestras de una distribución truncada por los límites originales
    de los datos, sin acumular probabilidad en los bordes.
    """
    low  = data.min()
    high = data.max()
    p_low  = dist.cdf(low)    # probabilidad acumulada hasta el límite inferior
    p_high = dist.cdf(high)   # probabilidad acumulada hasta el límite superior

    # Muestreamos uniforme SOLO en la franja de probabilidad válida
    u = np.random.uniform(p_low, p_high)

    # Transformada inversa: convierte probabilidades → valores
    return dist.ppf(u)

def generar_tc():
    """Generador estocástico de Tiempo de Carga para el motor de simulación."""
    return float(rvs_truncado(dist_tc, df["Charging Time (minutes)"]))

### 4.1. Tiempo de carga promedio (TCP) y factor de impaciencia (FI)

In [ ]:
# [PENDIENTE] TCP = E[TC] de la FDP truncada al rango observado; FI = 41.7 / TCP.

## 5. Obtención de las FDPs del Intervalo entre Arribos (IA)

El Intervalo de Arribos representa el tiempo entre que llega un vehículo al sistema y llega otro. Se utiliza la columna `Start Date` del dataset para calcular el tiempo entre llegadas, determinar la distribución estocástica que mejor represente este proceso, y poder generar valores aleatorios para dicha distribución.

> **[DESACTUALIZADO — TP 4]** El TP 4 ajusta un único `IA` global. El TP Final requiere seis FDPs (`IAS1`, `IAS2`, `IAS3`, `IAF1`, `IAF2`, `IAF3`) por franja horaria y tipo de día. Las celdas siguientes quedan como referencia del pipeline.

In [ ]:
# [DESACTUALIZADO — TP 4] IA global; reemplazar por las seis FDPs por franja.
df["Start Date"] = pd.to_datetime(df["Start Date"].dropna(), format="%m/%d/%Y %H:%M")
df = df.sort_values("Start Date")
df["Interarrival Time"] = df["Start Date"].diff().dt.total_seconds() / 60
df["Interarrival Time"] = df["Interarrival Time"].replace(0, 0.5)

data_arribos = df["Interarrival Time"].dropna()

mostrar_histograma(data_arribos, "Histograma del Intervalo de Arribos", "Intervalo de Arribos (minutos)")

Dado que esta distribución también tiene una cola derecha muy larga, se verá las distribuciónes que mejor se ajusten usando como criterio el `ks_statistic`.

In [ ]:
# [DESACTUALIZADO — TP 4] IA global; reemplazar por las seis FDPs por franja.
f_ia = mostrar_mejores_dist(data_arribos, "Intervalo de Arribos", "ks_statistic")

Se selecciona la distribución `Landau`, pues por más que no sea la de menor ks_statistic, no está lejos, y tiene menor sumsquare_error y menor kl_div, los cuales también son importantes de analizar para Simulación.

In [ ]:
# [DESACTUALIZADO — TP 4] IA global; reemplazar por las seis FDPs por franja.
dist_ia = obtener_dist(f_ia, "landau", "Intervalo de Arribos", data_arribos)

def generar_ia():
    """Generador estocástico de Intervalo de Arribos para el motor de simulación."""
    return float(rvs_truncado(dist_ia, data_arribos))

### 5.1. Segmentación por franja horaria y tipo de día

Franjas: 08:00–12:59, 13:00–19:59 y 20:00–07:59; días de semana y fin de semana.

In [ ]:
# [PENDIENTE] Segmentar los intervalos entre arribos por franja (según la hora del arribo)
# y por tipo de día, obteniendo las seis series IAS1..IAS3 e IAF1..IAF3.

### 5.2. Ajuste en paralelo de las seis FDPs

In [ ]:
# [PENDIENTE] Ajustar las seis series en paralelo (ProcessPoolExecutor, un worker por serie)
# con el mismo pipeline de fitter, y reportar ks_statistic, sumsquare_error y kl_div de cada una.

**Elección de las FDPs:** *[PENDIENTE] Una justificación por cada una de las seis series.*

## 6. Obtención de las FDPs de Falla (TFC) y Reparación (TRC) de Cargadores

`TFC` es el tiempo desde la puesta en servicio (reparación) de un cargador hasta su falla siguiente; `TRC`, el tiempo que permanece fuera de servicio hasta su reparación.

> **[PENDIENTE]** El dataset del TP 4 no contiene estos datos. Falta definir la fuente; si no se consigue un dataset, se adopta una FDP supuesta y se explicita la justificación.

In [ ]:
# [PENDIENTE] Carga de datos / ajuste de TFC y TRC y sus generadores.

## 7. Obtención del Costo Energético en función del TC

El Costo Energético representa el dinero gastado en la energía de los cargadores. Este depende de:
- El Tiempo de Carga (TC)
- El Precio de la Energía (PE): Valor constante que representa cuanto vale el kWh en CABA en Pesos

Para calcularlo, usaremos regresión lineal para obtener una función que nos permita pasar de TC a CE.


> **[DESACTUALIZADO — TP 4]** La regresión del TP 4 ordena por separado tiempo y energía (`np.sort`), lo que no es una regresión sobre los pares reales e infla el R². Rehacerla sobre los pares `(TC_k, kWh_k)` alineados y reportar el R² verdadero. El precio único `PE` se reemplaza por tarifas por franja (pico/valle). Además, la sigla `CE` del TP 4 (costo energético) choca con `CE` de la propuesta (cantidad de estaciones): renombrar.

In [ ]:
# [DESACTUALIZADO — TP 4] regresión sobre series ordenadas por separado; rehacer sobre pares alineados.
PE = 108.48 # Precio Promedio de Energia en CABA, en Pesos

tiempo = df["Charging Time (minutes)"].values
energia = df['Energy (kWh)'].dropna().values

slope, intercept, r, p, se = stats.linregress(
    np.sort(tiempo),
    np.sort(energia)
)

print(f"Energía [kWh] = {slope:.4f} x Tiempo [min] + {intercept:.4f}  (R²={r**2:.4f})")
print(f"Costo [ARS] = Energía [kWH] x {PE} [ARS/kWH]")

def tiempo_a_costo_lineal(t):
    return float((slope * t + intercept) * PE)

### 7.1. Tarifas de energía por franja horaria

In [ ]:
# [PENDIENTE] Precio de la energía por franja (pico/valle) con su fuente o supuesto explicitado.

## 8. Constantes y Parámetros del Modelo

Constantes según la propuesta: `CC_MAX`, `CE_MAX`, `CCP`, `CPN`, `CEN`, `TIC`, `TCE`, `PUNEN`, `FI`, `TCP`; y valores por defecto de las variables de control `RC`, `TMP` y `PDCE`.

> **[PENDIENTE]** Faltan los valores de `CC_MAX`, `CE_MAX`, `CPN`, `CEN`, `CCP` y los costos de mantenimiento. Cada supuesto se explicita con su justificación.

In [ ]:
# [PENDIENTE] Definición de constantes y valores por defecto de las variables de control.

## 9. Funciones Auxiliares

Reloj (mapeo de `T` a hora del día y tipo de día), generación del `IA` con la FDP vigente y el factor de crecimiento, prorrateo de acumuladores y costo de energía entre franjas, y decisión de arrepentimiento (`PUNEN`, `PU`, `EEU`).

In [ ]:
# [PENDIENTE] Funciones auxiliares del motor (a nivel de módulo, para que sean serializables).

def franja_y_tipo_dia(T):
    raise NotImplementedError("Pendiente: mapeo de T a franja y tipo de día (sección 9).")

def generar_ia(T):
    raise NotImplementedError("Pendiente: IA con la FDP vigente y el factor de crecimiento (sección 9).")

## 10. Implementación del Motor de Simulación

Se define un modelo de simulación de eventos discretos para un sistema multi-servidor (N y N) con lógica de abandono (arrepentimiento). El sistema gestiona una lista de eventos futuros basada en arribos y salidas por cada puesto.

> **[DESACTUALIZADO — TP 4]** Motor del TP 4: un solo IA global, elección aleatoria de puesto, arrepentimiento hardcodeado y estado en variables globales. Queda como referencia de la estructura del loop; se reemplaza por el motor de las subsecciones 10.1–10.3.

In [ ]:
# [DESACTUALIZADO — TP 4] motor de referencia; no usar para el TP Final.
# Variables de Control
N: int                        # Cantidad de puestos

#  Estado del reloj y eventos
T: float                      # Reloj actual
TF: float                     # Horizonte de simulación
TPLL: float                   # Próxima llegada global
TPS: List[float]              # Próxima salida por puesto

#  Estado del sistema
NS: List[int]                 # Autos en cada puesto (cola + servicio)

#  Contadores y acumuladores
CLL: int                      # Llegadas efectivas
CARR: int                     # Arrepentimientos
SPS: float                    # suma ponderada de personas en sistema
STC: float                    # Suma de tiempos de carga

ITO: List[float]              # Instante último cambio a ocioso por puesto
STO: List[float]              # Tiempo ocioso acumulado por puesto
SCE: List[float]              # Costo Energético acumulado por puesto

#  Resultados
PPS: float                    # Promedio de personas en sistema
PEC: float                    # Promedio en cola
PARR: float                   # % arrepentimiento
PTO: List[float]              # Proporción de ocio por puesto
CCE: List[float]              # Costo de Consumo Energetico por puesto

# Factor de comportamiento: probabilidad de que un usuario abandone la cola
PORCENTAJE_ARREPENTIMIENTO = 0.93

def condiciones_iniciales(cant_puestos: int):
    global TF, N
    global T, TPLL, TPS
    global NS
    global CLL, CARR, SPS, STC, ITO, STO, SCE

    # Variables de Control
    N = int(cant_puestos)

    # Reloj/eventos
    T = 0.0
    TF = 1000000.0
    TPLL = 0.0

    # Acumuladores
    CLL = 0
    CARR = 0
    SPS = 0.0
    STC = 0.0

    # Vectores
    TPS = [float("inf")] * N
    NS = [0] * N
    ITO = [0.0] * N
    STO = [0.0] * N
    SCE = [0.0] * N

def arrepentimiento(index_puesto) -> bool:
    global CARR, NS, PORCENTAJE_ARREPENTIMIENTO
    cant_autos = NS[index_puesto]

    if cant_autos <= 1:
        return False
    elif cant_autos <= 2:
        r = random.random()
        if r >= PORCENTAJE_ARREPENTIMIENTO:
            return False
        else:
            CARR += 1
            return True
    else:
        CARR += 1
        return True

def llegada():
    global TPLL, SPS, T, NS, CLL, STC, STO, ITO, TPS, N

    SPS += (TPLL - T) * (sum(NS))

    T = TPLL #Avanzamos en el tiempo

    IA = generar_ia() #Generamos el tiempo hasta la proxima llegada
    TPLL = T + IA #Calculamos el tiempo de llegada del proximo auto

    index_puesto = eleccion_server() #Subrutina
    if arrepentimiento(index_puesto):
        return

    # Llegada efectiva
    NS[index_puesto] += 1
    CLL += 1

    if NS[index_puesto] == 1:
        TC = generar_tc()
        STC += TC
        SCE[index_puesto] += tiempo_a_costo_lineal(TC)
        TPS[index_puesto] = T + TC
        STO[index_puesto] += (T - ITO[index_puesto])



def salida(index_puesto):
    global SPS, T, NS, STC, ITO, STO, TPS

    SPS += (TPS[index_puesto] - T) * (sum(NS))

    T = TPS[index_puesto]  # Avanzamos en el tiempo

    NS[index_puesto] -= 1  # Actualizo vector estado

    if NS[index_puesto] > 0:  # Evento Futuro Condicionado
        TC = generar_tc()
        STC += TC
        SCE[index_puesto] += tiempo_a_costo_lineal(TC)
        TPS[index_puesto] = T + TC  # Actualizo vector de tiempos de salida
    else:
        ITO[index_puesto] = T
        TPS[index_puesto] = float("inf")  # Actualizo vector de tiempos de salida

def eleccion_server():
    global N
    return random.randrange(N) # Distribución uniforme de la demanda

def menor_tps_index():
    global TPS
    return min(range(len(TPS)), key=TPS.__getitem__)

def calculo_resultados():
    global SPS, STC, SCE, CLL, CARR, STO, T, N
    total_arribos = CLL + CARR

    PPS = SPS / CLL if CLL > 0 else 0.0
    PEC = (SPS - STC) / CLL if CLL > 0 else 0.0
    PARR = CARR * 100 / total_arribos if total_arribos > 0 else 0.0

    PTO = []
    CCE = []

    for i in range(N):
        PTO.append(STO[i] * 100 / T if T > 0 else 0.0)

    for i in range(N):
        CCE.append(SCE[i] * 100 / CLL if CLL > 0 else 0.0)

    print(f"PPS = {round(PPS,2)}")
    print(f"PEC = {round(PEC,2)}")
    print(f"PTO = {[round(x, 2) for x in PTO]}")
    print(f"CCE = {[round(x, 2) for x in CCE]}")
    print(f"PARR = {round(PARR,2)}")


def simular(cant_puestos: int):
    global TPLL, TPS, T, TF, NS

    condiciones_iniciales(cant_puestos)

    while True:
        index_server = menor_tps_index()

        if TPLL <= TPS[index_server]:
            llegada()
        else:
            salida(index_server)

        if T <= TF:
            continue
        else: # Vaciamiento
            if any(x > 0 for x in NS):
                TPLL = float("inf")
                continue
            break
    calculo_resultados()

### 10.1. Estado de la simulación

In [ ]:
# [PENDIENTE] Estado encapsulado (dataclass): variables de estado CA(i), CD(i), CC(i), CE;
# TEF TPI(i), TPC(i)(j), TPAE, TPIC(i), TPCE, TPFC(i)(j), TPRC(i)(j), TPMP(i);
# auxiliares CARRUM(i), CPAACUM, CAPF(i), ICC(i)(j), CTC(i)(j) y acumuladores por franja.

### 10.2. Rutinas de eventos

Una rutina por evento de la tabla de eventos y condiciones de la propuesta.

In [ ]:
# [PENDIENTE] Rutinas: Ingreso de auto a estación (i), Carga en cargador (i)(j),
# Análisis de Expansión, Instalación de nuevo cargador (i), Construcción de nueva estación,
# Falla de cargador (i)(j), Reparación de cargador (i)(j), Mantenimiento preventivo (i).

### 10.3. Función de simulación

Función pura: recibe parámetros y semilla, devuelve las métricas de resultado (`BM`, `PTO`, `PEC`, `PPS`, `PARR`, `PDC`, `PAPF` por franja y sus promedios).

In [ ]:
# [PENDIENTE] Loop evento a evento (avance al mínimo de la TEF) y cálculo de resultados.

def simular_tp_final(RC, TMP, PDCE, TF, semilla):
    raise NotImplementedError("Pendiente: motor del TP Final (sección 10.3).")

## 11. Verificación y Validación del Motor

Invariantes y aserciones derivados de la tabla de eventos y condiciones de la propuesta, y validación contra el modelo analítico M/M/c en un escenario reducido.

In [ ]:
# [PENDIENTE] Chequeos sobre valores y cambios de estado (no sobre "no tiró excepción"),
# p. ej. autos cargando = min(CA(i), CD(i)), y comparación contra M/M/c.

## 12. Experimentación y Análisis de Sensibilidad

Se ejecutan múltiples escenarios variando el número de puestos de carga (N) para identificar el punto de equilibrio operativo donde se minimiza el arrepentimiento y el tiempo de espera en cola, manteniendo una utilización eficiente del cargador.

> **[DESACTUALIZADO — TP 4]** El barrido del TP 4 varía la cantidad de puestos `N`. En el TP Final la cantidad de cargadores y estaciones es dinámica; los escenarios varían `RC`, `TMP` y `PDCE`.

In [ ]:
# [DESACTUALIZADO — TP 4] barrido sobre N; reemplazar por la grilla de RC, TMP y PDCE.
# Simulación para configuraciones de 1 a 40 puestos
for i in range(1, 41):
    print(f"\n--- ESCENARIO: N = {i} puestos ---")
    simular(cant_puestos=i)

### 12.1. Grilla de escenarios y réplicas en paralelo

In [ ]:
# [PENDIENTE] Grilla de combinaciones (RC, TMP, PDCE) × réplicas, ejecutada con
# ProcessPoolExecutor (workers según os.cpu_count()), con semilla por corrida derivada
# de una semilla base + índice de combinación/réplica.

### 12.2. Análisis de resultados

In [ ]:
# [PENDIENTE] Tablas y gráficos comparativos de las métricas por escenario, con intervalos de confianza.

## 13. Conclusiones

*[PENDIENTE]*